# BRAMASTRA K8: Two-T4 Cognition and Recursive Improvement Campaign

**Owner-launched.** Allocation: max **480 elapsed min / 960 GPU-min** on 2x T4.

**Important:** kernel restarts do NOT reset the clock (SQLite ledger).
If E0 already ran, the full-run cell skips it.
Stop new training by minute 450. Hard stop before 480.

In [ ]:
import json, os, subprocess, sys
REPO = '/kaggle/working/An-Ra-the-new-AGI'
if os.path.isdir(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
import torch
from bramastra_lab.research.runtime.provenance import source_identity
src = source_identity()
print('source:', json.dumps(src, indent=2))
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  cuda:{i}:', torch.cuda.get_device_name(i))
print('Phases: E0(0-30) E1(30-150) E2(150-195) E3(195-255) E4(255-315) E5(315-450) E6(450-480)')


## 2. Prepare Data (idempotent)

In [ ]:
BUNDLE_DIR = '/kaggle/working/bramastra-k8-data'
RUN_DIR = '/kaggle/working/K8-campaign'
result = subprocess.run(
    [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
     'prepare', '--out', BUNDLE_DIR,
     '--training-mechanisms', '4096', '--controller-mechanisms', '256',
     '--development-mechanisms', '256', '--confirmation-mechanisms', '128',
     '--tool-mechanisms', '256', '--tool-heldout', '64',
     '--meta-train', '24', '--meta-validate', '6', '--meta-confirm', '6'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError('prepare failed')


## 3. Validate Data

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
     'validate', '--bundle', BUNDLE_DIR],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError('validate failed')


## 4. E0 Gate

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
     'run', '--mode', 'e0', '--run-dir', RUN_DIR,
     '--data', BUNDLE_DIR, '--max-wall-minutes', '480',
     '--precision', 'fp16_autocast'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError('E0 gate failed')


## 5. Full Campaign (E0 through E6)
If E0 already ran, this cell skips it and uses remaining time.

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
     'run', '--mode', 'full', '--run-dir', RUN_DIR,
     '--data', BUNDLE_DIR, '--max-wall-minutes', '480',
     '--precision', 'fp16_autocast'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError('full campaign failed')


## 6. Summarize

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
     'summarize', '--run-dir', RUN_DIR],
    capture_output=True, text=True)
print(result.stdout)


## 7. Export

In [ ]:
EXPORT_DIR = '/kaggle/working/K8-results'
result = subprocess.run(
    [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
     'export', '--run-dir', RUN_DIR, '--out', EXPORT_DIR],
    capture_output=True, text=True)
print(result.stdout)
print('Copy /kaggle/working/K8-results/ to persistent storage.')
